In [4]:
import pandas as pd
import numpy as np

# World Happiness Report 2023 — representative data
# Source: https://www.kaggle.com/datasets/ajaypalsinghlo/world-happiness-report-2023

df = pd.read_csv('/kaggle/input/datasets/sachinarkesh/world-happiness-2023/world_happiness_2023.csv')
df.columns = ['Country','Region','Happiness_Score','GDP','Social_Support',
              'Life_Expectancy','Freedom','Generosity','Corruption']


print(f"Dataset: {len(df)} countries, {len(df.columns)} columns")
print(df.head())

Dataset: 63 countries, 9 columns
       Country                        Region  Happiness_Score     GDP  \
0      Finland                Western Europe            7.804  10.775   
1      Denmark                Western Europe            7.586  10.933   
2      Iceland                Western Europe            7.525  10.878   
3       Israel  Middle East and North Africa            7.473  10.527   
4  Netherlands                Western Europe            7.464  11.015   

   Social_Support  Life_Expectancy  Freedom  Generosity  Corruption  
0           0.954             71.9    0.949       0.142       0.179  
1           0.954             72.7    0.931       0.168       0.234  
2           0.983             72.5    0.961       0.260       0.150  
3           0.916             72.4    0.903       0.149       0.826  
4           0.939             72.4    0.879       0.240       0.296  


In [5]:
import plotly.express as px
import plotly.graph_objects as go

# Explore the dataset before you start
print("Regions in dataset:")
print(df['Region'].value_counts())
print("\nScore range:", df['Happiness_Score'].min(), "–", df['Happiness_Score'].max())
print("\nBottom 10 countries:")
print(df.nsmallest(10, 'Happiness_Score')[['Country','Region','Happiness_Score']])

Regions in dataset:
Region
Western Europe                  15
Latin America and Caribbean     13
Central and Eastern Europe       7
Sub-Saharan Africa               7
Middle East and North Africa     6
North America and ANZ            4
Southeast Asia                   4
South Asia                       4
East Asia                        3
Name: count, dtype: int64

Score range: 1.859 – 7.804

Bottom 10 countries:
        Country                        Region  Happiness_Score
60  Afghanistan                    South Asia            1.859
61      Lebanon  Middle East and North Africa            2.392
62     Zimbabwe            Sub-Saharan Africa            2.995
52     Ethiopia            Sub-Saharan Africa            3.564
53     Tanzania            Sub-Saharan Africa            3.698
48   Bangladesh                    South Asia            3.892
47        India                    South Asia            4.036
50        Kenya            Sub-Saharan Africa            4.112
54       Uganda

In [7]:
# Task 1: Regional comparison bar chart
# -------------------------------------

# Step 1: Compute average happiness score by region
region_avg = (df.groupby('Region')['Happiness_Score']
              .mean()
              .reset_index()
              .sort_values('Happiness_Score'))  # ascending -> happiest ends up on top in a horizontal bar

print(region_avg)

# Step 2: Build your chart
global_avg = df['Happiness_Score'].mean()
best = region_avg.iloc[-1]
worst = region_avg.iloc[0]

# Colour: highlight the happiest region, mute the rest, flag the least happy
GREY, BLUE, RED = '#B8BEC6', '#1F6FB2', '#C0392B'
bar_colors = [RED if r == worst['Region'] else BLUE if r == best['Region'] else GREY
              for r in region_avg['Region']]

fig1 = go.Figure(go.Bar(
    x=region_avg['Happiness_Score'],
    y=region_avg['Region'],
    orientation='h',
    marker_color=bar_colors,
    text=region_avg['Happiness_Score'].round(2),
    textposition='outside',
    cliponaxis=False,
    hovertemplate='%{y}: %{x:.2f}<extra></extra>',
))

# Global-average reference line
fig1.add_vline(x=global_avg, line_dash='dash', line_color='#444', line_width=1.5,
               annotation_text=f'Global avg {global_avg:.2f}',
               annotation_position='top', annotation_font_size=12)

fig1.update_layout(
    title=dict(
        text=(f"<b>{best['Region']} is the happiest region ({best['Happiness_Score']:.1f}), "
              f"nearly {best['Happiness_Score'] - worst['Happiness_Score']:.1f} points ahead of "
              f"{worst['Region']} ({worst['Happiness_Score']:.1f})</b>"),
        x=0, xanchor='left', font=dict(size=18)),
    xaxis=dict(title='Average happiness score (0-10 scale)', range=[0, 10],  # zero baseline
               showgrid=True, gridcolor='#ECECEC', zeroline=False),
    yaxis=dict(title='', automargin=True),
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial, Helvetica, sans-serif', size=13, color='#222'),
    margin=dict(l=20, r=40, t=90, b=60),
    height=520, width=950, showlegend=False,
)
fig1.add_annotation(text='Source: World Happiness Report 2023', xref='paper', yref='paper',
                    x=0, y=-0.14, showarrow=False, font=dict(size=11, color='#777'))
fig1.show()

                         Region  Happiness_Score
5                    South Asia         3.618250
7            Sub-Saharan Africa         4.064714
3  Middle East and North Africa         4.943333
6                Southeast Asia         5.695250
2   Latin America and Caribbean         5.699000
1                     East Asia         5.966000
0    Central and Eastern Europe         6.338143
4         North America and ANZ         7.018250
8                Western Europe         7.085533


In [8]:
# Task 2: Top 8 vs. Bottom 8 contrast
# ------------------------------------

# Step 1: Get top and bottom countries
top8 = df.nlargest(8, 'Happiness_Score').copy()
top8['Group'] = 'Top 8'
bottom8 = df.nsmallest(8, 'Happiness_Score').copy()
bottom8['Group'] = 'Bottom 8'

combined = pd.concat([bottom8, top8]).sort_values('Happiness_Score')
global_avg = df['Happiness_Score'].mean()
print(f"Global average: {global_avg:.2f}")

# Step 2: Build your chart
top_mean = top8['Happiness_Score'].mean()
bottom_mean = bottom8['Happiness_Score'].mean()
gap = top_mean - bottom_mean

group_colors = {'Top 8': '#1F6FB2', 'Bottom 8': '#C0392B'}

fig2 = go.Figure()
for group in ['Top 8', 'Bottom 8']:
    sub = combined[combined['Group'] == group]
    fig2.add_trace(go.Bar(
        x=sub['Happiness_Score'], y=sub['Country'], orientation='h',
        name=group, marker_color=group_colors[group],
        text=sub['Happiness_Score'].round(2), textposition='outside', cliponaxis=False,
        hovertemplate='%{y}: %{x:.2f}<extra></extra>',
    ))

# Keep the sorted order (lowest at bottom, highest at top)
fig2.update_yaxes(categoryorder='array', categoryarray=combined['Country'].tolist())

# Visual separator between the two groups (between bar 8 and bar 9 -> y = 7.5)
fig2.add_hline(y=7.5, line_dash='dot', line_color='#888', line_width=1.5)

# Stretch goal: global average reference line
fig2.add_vline(x=global_avg, line_dash='dash', line_color='#444', line_width=1.5,
               annotation_text=f'Global avg {global_avg:.2f}', annotation_position='top',
               annotation_font_size=12)

# Annotation that emphasises the gap
fig2.add_annotation(
    x=top_mean * 0.55, y=7.5, xref='x', yref='y',
    text=f'<b>The gap: {gap:.1f} points</b><br>(avg {top_mean:.1f} vs {bottom_mean:.1f})',
    showarrow=False, bgcolor='white', bordercolor='#888', borderwidth=1, borderpad=6,
    font=dict(size=13),
)

fig2.update_layout(
    title=dict(
        text=(f"<b>The happiest countries score {top_mean / bottom_mean:.1f}x higher than the least happy "
              f"- a {gap:.1f}-point gap</b>"),
        x=0, xanchor='left', font=dict(size=18)),
    xaxis=dict(title='Happiness score (0-10 scale)', range=[0, 10],  # zero baseline
               showgrid=True, gridcolor='#ECECEC', zeroline=False),
    yaxis=dict(title='', automargin=True),
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial, Helvetica, sans-serif', size=13, color='#222'),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    margin=dict(l=20, r=40, t=100, b=60),
    height=720, width=950, bargap=0.25,
)
fig2.add_annotation(text='Source: World Happiness Report 2023', xref='paper', yref='paper',
                    x=0, y=-0.09, showarrow=False, font=dict(size=11, color='#777'))
fig2.show()

Global average: 5.81


In [9]:
# Stretch goal — grouped bar chart
# Note: the column is named 'GDP' after the rename in the setup cell (not 'GDP_per_capita').

wanted = ['Western Europe', 'Latin America', 'East Asia', 'Sub-Saharan Africa', 'South Asia']
available = set(df['Region'])
regions = [r for r in wanted if r in available]
if len(regions) < 5:  # region labels differ between dataset versions -> fall back to the 5 largest regions
    regions = df['Region'].value_counts().head(5).index.tolist()
    print("Requested region names not all found; using the 5 largest regions:", regions)

factors = df[df['Region'].isin(regions)].groupby('Region')[['GDP', 'Freedom']].mean()
factors = factors.sort_values('GDP', ascending=False)

fig3 = go.Figure()
fig3.add_trace(go.Bar(x=factors.index, y=factors['GDP'], name='GDP (economic output)',
                      marker_color='#1F6FB2', text=factors['GDP'].round(2), textposition='outside'))
fig3.add_trace(go.Bar(x=factors.index, y=factors['Freedom'], name='Freedom to make life choices',
                      marker_color='#E69F00', text=factors['Freedom'].round(2), textposition='outside'))

top_gdp, low_gdp = factors['GDP'].idxmax(), factors['GDP'].idxmin()
fig3.update_layout(
    barmode='group',
    title=dict(
        text=(f"<b>Wealth varies far more across regions than freedom does: "
              f"{top_gdp} leads GDP by {factors['GDP'].max() / factors['GDP'].min():.1f}x over {low_gdp}, "
              f"yet freedom scores sit much closer together</b>"),
        x=0, xanchor='left', font=dict(size=16)),
    yaxis=dict(title='Average contribution to happiness score', rangemode='tozero',  # zero baseline
               showgrid=True, gridcolor='#ECECEC', zeroline=False),
    xaxis=dict(title=''),
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial, Helvetica, sans-serif', size=13, color='#222'),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    margin=dict(l=20, r=40, t=120, b=60),
    height=560, width=950, bargap=0.25,
)
fig3.show()

Requested region names not all found; using the 5 largest regions: ['Western Europe', 'Latin America and Caribbean', 'Central and Eastern Europe', 'Sub-Saharan Africa', 'Middle East and North Africa']
